# Demo: Bayesian Hyperparameter Optimisation with MyBO

**Author:** Nikolas Karefyllidis, PhD  
**Context:** ICL PCMLAI BBO Capstone — NeurIPS 2020 competition format

---

## What this notebook shows

This notebook applies the `BayesianOptimizer` class built for the BBO challenge to a
**real machine learning task**: tuning a `RandomForestClassifier` on sklearn's
**Digits dataset** (10-class digit recognition, 64 features, 1797 samples).

The same GP surrogate, LML kernel selection, and EI+PI+UCB ensemble acquisition
used in the 13-week BBO challenge are applied here with a clean `suggest / observe` API.

**Search space (4D):**

| Parameter | Range | Type |
|---|---|---|
| `n_estimators` | [10, 500] | int |
| `max_depth` | [2, 30] | int |
| `min_samples_split` | [0.01, 0.5] | float |
| `max_features` | [0.1, 0.99] | float |

**Objective:** 5-fold stratified CV accuracy (maximise).  
**Budget:** 10 LHS warm-start + 20 BO iterations = **30 total evaluations**.  
**Baseline:** Random search with the same 30-evaluation budget.

---

### Why Bayesian Optimisation for HPO?

Each evaluation requires fitting and cross-validating a model — expensive at scale.
BO uses a **GP surrogate** to predict which hyperparameter configurations are likely
to be good *before* evaluating them, guided by the acquisition function's
exploration-exploitation trade-off. This is sample-efficient: it finds good
configurations faster than random or grid search.

## 1. Setup

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.metrics import classification_report
from skopt.sampler import Lhs

# Ensure project root is on path (works from notebooks/ and from project root)
repo_root = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(repo_root))

from src.optimizers.optimizer import BayesianOptimizer

print(f"Project root: {repo_root}")
print(f"BayesianOptimizer imported from: {BayesianOptimizer.__module__}")

## 2. Dataset and search space

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
data = load_digits()
X_data, y_data = data.data, data.target
print(f"Dataset: Digits ({X_data.shape[0]} samples, {X_data.shape[1]} features, "
      f"{len(data.target_names)} classes)")

# ── Search space ─────────────────────────────────────────────────────────────
# Bounds for the 4 RandomForestClassifier hyperparameters.
# Integer-valued params (n_estimators, max_depth) are treated as continuous
# here and rounded to int inside the objective — standard practice for GP-BO.
BOUNDS = [
    (10.0,  500.0),   # n_estimators
    ( 2.0,   30.0),   # max_depth
    ( 0.01,   0.5),   # min_samples_split (fraction)
    ( 0.1,   0.99),   # max_features (fraction)
]
DIM_NAMES = ["n_estimators", "max_depth", "min_samples_split", "max_features"]

print("\nSearch space (4D):")
for name, (lo, hi) in zip(DIM_NAMES, BOUNDS):
    print(f"  {name:22s}: [{lo}, {hi}]")

## 3. Objective function

The objective maps a hyperparameter vector `x ∈ [lo, hi]^4` to the
5-fold stratified cross-validated accuracy of a `RandomForestClassifier`.
This is expensive to evaluate — exactly the regime where BO adds value.

In [ ]:
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def objective(x: np.ndarray) -> float:
    """5-fold CV accuracy of RandomForestClassifier on Digits. Maximize."""
    clf = RandomForestClassifier(
        n_estimators=max(10, int(round(x[0]))),
        max_depth=max(2, int(round(x[1]))),
        min_samples_split=float(np.clip(x[2], 0.01, 0.5)),
        max_features=float(np.clip(x[3], 0.1, 0.99)),
        random_state=42,
        n_jobs=-1,
    )
    scores = cross_val_score(clf, X_data, y_data, cv=cv_splitter, scoring="accuracy")
    return float(scores.mean())


# Sanity-check: evaluate at a reasonable default configuration
x_default = np.array([100.0, 10.0, 0.02, 0.5])
y_default = objective(x_default)
print(f"Default RF (n_est=100, depth=10, mss=0.02, mf=0.5):  CV acc = {y_default:.4f}")
print("Objective callable is working. Starting optimization.")

## 4. Bayesian Optimisation with MyBO

### Strategy

1. **Warm-start (10 points):** Latin Hypercube Sampling (LHS) fills the 4D space evenly.
   These give the GP enough data to form an initial surrogate.
2. **BO loop (20 iterations):** At each step:
   - Fit 3 GP kernels (RBF, Matérn, RBF+WhiteKernel); select the best by LML.
   - Compute EI, PI, UCB over a Sobol candidate pool.
   - **Ensemble decision:** if the three argmaxes agree (max pairwise L2 < 0.22),
     use the EI point; otherwise use the centroid (softens over-commitment).
   - Evaluate the selected point and record `(x, y)`.

The `BayesianOptimizer` API is a thin stateful wrapper around the same `suggest()`
function used for the 13-week challenge.

In [ ]:
N_INIT = 10   # LHS warm-start evaluations
N_ITER = 20   # Bayesian optimisation iterations
TOTAL  = N_INIT + N_ITER   # 30 total

# Build LHS warm-start points in original bounds
lhs_sampler = Lhs(criterion="maximin", iterations=50)
X_init_01 = np.array(lhs_sampler.generate([(0.0, 1.0)] * 4, N_INIT))
X_init = np.zeros_like(X_init_01)
for j, (lo, hi) in enumerate(BOUNDS):
    X_init[:, j] = lo + X_init_01[:, j] * (hi - lo)

# Instantiate optimizer — same settings as late-round BBO challenge configs
opt = BayesianOptimizer(
    bounds=BOUNDS,
    xi=0.01,            # EI/PI exploration parameter (low = exploit-leaning)
    kappa=0.75,         # UCB exploration parameter
    use_ensemble=True,
    agree_threshold=0.22,
    candidate_method="sobol",
    seed=42,
)
print(opt)

print(f"\nWarm-start: {N_INIT} LHS evaluations")
bo_curve = []   # incumbent best at each evaluation
for i, x in enumerate(X_init):
    y = objective(x)
    opt.observe(x, y)
    bo_curve.append(opt.best_y)
    print(f"  init {i + 1:2d}/{N_INIT}: acc={y:.4f}  best={opt.best_y:.4f}")

In [ ]:
print(f"Bayesian Optimisation: {N_ITER} iterations")
for i in range(N_ITER):
    x_next = opt.suggest()          # GP + ensemble acquisition
    y_next = objective(x_next)      # evaluate the objective (expensive)
    opt.observe(x_next, y_next)     # update the internal observation set
    bo_curve.append(opt.best_y)
    print(f"  iter {i + 1:2d}/{N_ITER}: acc={y_next:.4f}  best={opt.best_y:.4f}")

print(f"\nMyBO finished. {len(opt)} total observations, best_y = {opt.best_y:.4f}")

## 5. Random Search baseline

Same total budget (30 evaluations), uniform random sampling across the bounds.
No surrogate model — every point is chosen without any knowledge of prior
evaluations.  This is the standard baseline for benchmarking BO.

In [ ]:
print(f"Random Search: {TOTAL} evaluations (same budget as MyBO)")
rng = np.random.default_rng(42)
rand_ys: list[float] = []
rand_curve: list[float] = []

for i in range(TOTAL):
    x_r = np.array([rng.uniform(lo, hi) for lo, hi in BOUNDS])
    y_r = objective(x_r)
    rand_ys.append(y_r)
    rand_curve.append(float(np.max(rand_ys)))
    if (i + 1) % 5 == 0:
        print(f"  {i + 1:2d}/{TOTAL}: acc={y_r:.4f}  best={rand_curve[-1]:.4f}")

print(f"\nRandom Search finished. Best CV accuracy = {rand_curve[-1]:.4f}")

## 6. Results

In [ ]:
evals = np.arange(1, TOTAL + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    "Bayesian Optimisation vs Random Search\n"
    "(RandomForestClassifier, Digits dataset, 30 evaluations)",
    fontsize=13,
)

# ── Left: convergence (incumbent accuracy vs evaluations) ────────────────────
axes[0].axvspan(0.5, N_INIT + 0.5, alpha=0.07, color="green", zorder=0)
axes[0].axvline(N_INIT + 0.5, color="green", linewidth=1.5, linestyle=":",
                alpha=0.6, label=f"BO start (eval {N_INIT + 1})")
axes[0].plot(evals, bo_curve, "b-o",
             label="MyBO  (GP + EI/PI/UCB ensemble)",
             linewidth=2, markersize=5, zorder=3)
axes[0].plot(evals, rand_curve, "r--s",
             label="Random Search",
             linewidth=2, markersize=5, alpha=0.85, zorder=2)

y_min = max(0.85, min(bo_curve + rand_curve) - 0.01)
axes[0].set_ylim([y_min, 1.005])
axes[0].set_xlabel("Number of evaluations", fontsize=11)
axes[0].set_ylabel("Best CV accuracy (5-fold)", fontsize=11)
axes[0].set_title("Convergence: best-so-far accuracy")
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# ── Right: search trajectory (n_estimators vs max_depth, colour = CV acc) ───
X_obs = opt.X
y_obs = opt.y
sc = axes[1].scatter(
    X_obs[:, 0], X_obs[:, 1], c=y_obs,
    cmap="viridis", s=80, edgecolors="k", linewidths=0.4, zorder=3,
)
plt.colorbar(sc, ax=axes[1], label="CV Accuracy")
best_x, best_y_val = opt.best
axes[1].scatter(
    [best_x[0]], [best_x[1]],
    marker="*", s=450, c="red", zorder=5,
    label=f"Best  acc={best_y_val:.4f}",
)
# Annotate warm-start vs BO points
axes[1].scatter(
    X_obs[:N_INIT, 0], X_obs[:N_INIT, 1],
    marker="^", s=80, facecolors="none", edgecolors="green",
    linewidths=1.5, zorder=4, label="LHS warm-start",
)
axes[1].set_xlabel("n_estimators", fontsize=11)
axes[1].set_ylabel("max_depth", fontsize=11)
axes[1].set_title("MyBO search trajectory\n(n_estimators vs max_depth, colour = CV acc)")
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = repo_root / "docs" / "demo_rf_hpo_convergence.png"
plt.savefig(plot_path, dpi=120, bbox_inches="tight")
print(f"Plot saved to {plot_path.relative_to(repo_root)}")
plt.show()

In [ ]:
best_x, best_y_val = opt.best
rand_best = rand_curve[-1]
gain_pp = (best_y_val - rand_best) * 100

print("=" * 55)
print("Results summary")
print("=" * 55)
print(f"  MyBO   best CV accuracy : {best_y_val:.4f}")
print(f"  Random best CV accuracy : {rand_best:.4f}")
print(f"  MyBO improvement        : {gain_pp:+.2f} percentage points")
print()
print("Best hyperparameters found by MyBO:")
for name, val in zip(DIM_NAMES, best_x):
    if name in ("n_estimators", "max_depth"):
        print(f"  {name:22s}: {int(round(val))}")
    else:
        print(f"  {name:22s}: {val:.4f}")

## 7. Final model evaluation

Train the best configuration found by MyBO on 80% of the data and evaluate
on a held-out 20% test set.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

clf_bo = RandomForestClassifier(
    n_estimators=max(10, int(round(best_x[0]))),
    max_depth=max(2, int(round(best_x[1]))),
    min_samples_split=float(np.clip(best_x[2], 0.01, 0.5)),
    max_features=float(np.clip(best_x[3], 0.1, 0.99)),
    random_state=42,
    n_jobs=-1,
)
clf_bo.fit(X_train, y_train)
test_acc_bo = clf_bo.score(X_test, y_test)

print(f"Final model (MyBO best config) — test accuracy: {test_acc_bo:.4f}")
print()
print(classification_report(y_test, clf_bo.predict(X_test),
                             target_names=[str(c) for c in data.target_names]))

## 8. Summary

### What happened

- The GP surrogate learned which regions of the 4D hyperparameter space tend
  to produce high accuracy, and the ensemble acquisition directed subsequent
  queries there — getting more out of the same evaluation budget compared to
  random search.
- The `BayesianOptimizer` class provides a clean `suggest / observe` loop that
  can be dropped into any Python HPO workflow.

### Connections to the BBO challenge

| Challenge component | This demo |
|---|---|
| Unknown oracle function | 5-fold CV of RF (treat as black-box) |
| 4D search space | `[n_estimators, max_depth, min_samples_split, max_features]` |
| GP surrogate (LML kernel selection) | Same `my_gp_skopt.suggest()` internals |
| EI + PI + UCB ensemble | Same `AGREE_THRESHOLD = 0.22` logic |
| LHS warm-start | Same Sobol/LHS candidate sampling |
| Challenge Function 7 | Analogue: this notebook is exactly that function |

### Real-world relevance

The same technique scales to:
- **Neural architecture search** — treating layer widths, learning rates, dropout
  as continuous parameters.
- **Drug discovery / materials science** — experimental design where each
  measurement is costly.
- **A/B test design** — sequential allocation of traffic to treatment arms.
- **AutoML pipelines** — any setting where evaluation is expensive and
  the objective is smooth enough to be modelled by a GP.